# BioASQ 14b pooled-gold pilot
Restricted PubMed corpus; **not** the official full-PubMed leaderboard benchmark. The fixed pilot contains 200 questions selected with seed `20261028`.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Markdown
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'artifacts/external-benchmarks/bioasq14b').exists())
BASE = ROOT / 'artifacts/external-benchmarks/bioasq14b'
coverage = json.loads((BASE / 'coverage_and_selection.json').read_text())
summary = json.loads((BASE / 'pilot/benchmark_summary.json').read_text())
rows = [json.loads(line) for line in (BASE / 'pilot/per_query_rankings.jsonl').read_text().splitlines() if line]
by_query = {row['query_id']: row for row in rows}
display(pd.DataFrame([coverage['summary']]).T.rename(columns={0:'value'}))
display(Markdown('> ' + summary['benchmark_label']))

In [ ]:
display(pd.DataFrame(summary['summary']).T.style.format('{:.4f}'))

In [ ]:
def show_question(query_id):
    row = by_query[query_id]
    display(Markdown(f"## {query_id}\n{row['question']}\n\nType: `{row['question_type']}` · gold coverage: `{row['gold_coverage']:.1%}`"))
    display(pd.DataFrame({'gold_pmid': row['gold_pmids'], 'available': [p in row['available_gold_pmids'] for p in row['gold_pmids']]}))
    for method, result in row['methods'].items():
        display(Markdown(f"### {method.upper()} — {result['latency_ms']:.2f} ms"))
        frame = pd.DataFrame(result['ranking'][:20])
        frame.insert(0, 'rank', range(1, len(frame)+1))
        frame['gold'] = frame['pmid'].isin(row['gold_pmids'])
        display(frame)
        display(pd.DataFrame([result['metrics']]))
query_ids = list(by_query)
show_question(query_ids[0])

In [ ]:
try:
    import ipywidgets as widgets
    widgets.interact(show_question, query_id=widgets.Dropdown(options=query_ids, description='Question'))
except ImportError:
    print('Call show_question(query_ids[0]) manually.')